# Ch8-1 Playground — 叉積的標準介��

> 對應 3Blue1Brown《線性代數的本質》第八章（上）
>
> **上一章回顧**：點積把兩個向量變成一個**純量**（數字），衡量「方向有多像」。
>
> **這一章**：叉積把兩個向量變成一個**新向量**，衡量「兩個向量圍出多大面積、往哪個方向」。

---

## 本章路線圖

| Part | 主題 | 核心直覺 |
|---|---|---|
| 1 | 2D 叉積（偽叉積） | 兩個 2D 向量圍成的平行四邊形面積，帶正負號 |
| 2 | 面積 = 行列式！ | 把兩個向量排成矩陣的 column，det 就是帶符號面積 |
| 3 | 3D 叉積 | 結果是一個**向量**：大小 = 面積，方向 = 垂直於兩者 |
| 4 | 右手定則 | 叉積方向怎麼判斷 |
| 5 | 3D 叉積公式 & 計算 | 那個「假行列式」展開法 |
| 6 | 動手實驗 | 自己玩 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

%matplotlib inline
plt.rcParams['figure.figsize'] = (7, 7)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

print("Ready!")

---
## Part 1：2D「叉積」— 平行四邊形的帶符號面積

### 理論

嚴格來說，叉積只定義在 3D。但 3Blue1Brown 先從 2D 講起，因為直覺更清楚。

兩個 2D 向量 $\vec{v}$ 和 $\vec{w}$ 圍成一個平行四邊形，它的**帶符號面積**是：

$$\vec{v} \times \vec{w} = v_1 w_2 - v_2 w_1$$

這其實就是把 $\vec{v}$、$\vec{w}$ 當成矩陣的兩個 **column**，然後算 **det**！

$$\det \begin{bmatrix} v_1 & w_1 \\ v_2 & w_2 \end{bmatrix} = v_1 w_2 - v_2 w_1$$

正負號的意義：
- **正**：$\vec{w}$ 在 $\vec{v}$ 的**逆時針方向**（左手邊）
- **負**：$\vec{w}$ 在 $\vec{v}$ 的**順時針方向**（右手邊）
- **零**：兩向量平行（面積為零）

In [ ]:
# === 實驗 1：2D 叉積 = 平行四邊形的帶符號面積 ===

def plot_2d_cross(v, w, ax=None):
    """畫出兩個 2D 向量圍成的平行四邊形，標記面積"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))
    
    v, w = np.array(v, dtype=float), np.array(w, dtype=float)
    cross_2d = v[0]*w[1] - v[1]*w[0]  # det
    
    # 平行四邊形的四個頂點：O, v, v+w, w
    parallelogram = np.array([
        [0, 0], v, v + w, w
    ])
    
    # 根據正負號選顏色
    color = 'lightcoral' if cross_2d >= 0 else 'lightblue'
    sign_text = '(+) 逆時針' if cross_2d > 0 else ('(−) 順時針' if cross_2d < 0 else '= 0 平行!')
    
    # 畫平行四邊形
    ax.add_patch(plt.Polygon(parallelogram, alpha=0.35, color=color,
                              edgecolor='gray', lw=1.5))
    
    # 畫向量
    ax.annotate('', xy=v, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
    ax.annotate('', xy=w, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
    
    # 標籤
    ax.text(v[0]*0.5 + 0.15, v[1]*0.5 + 0.15, f'v=({v[0]:.0f},{v[1]:.0f})', 
            fontsize=11, color='red', fontweight='bold')
    ax.text(w[0]*0.5 - 0.3, w[1]*0.5 + 0.15, f'w=({w[0]:.0f},{w[1]:.0f})',
            fontsize=11, color='blue', fontweight='bold')
    
    # 面積數值放在平行四邊形中心
    center = (v + w) / 2
    ax.text(center[0], center[1], f'Area = {abs(cross_2d):.1f}',
            fontsize=13, ha='center', va='center', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    lim = max(np.abs(np.concatenate([v, w, v+w]))) + 1.5
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_title(f"v × w = {cross_2d:.1f}  {sign_text}", fontsize=13)
    return ax

# 三種情況並排
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

plot_2d_cross([3, 0], [1, 2], ax=axes[0])     # 正（逆時針）
plot_2d_cross([3, 0], [1, -2], ax=axes[1])     # 負（順時針）
plot_2d_cross([2, 1], [4, 2], ax=axes[2])      # 零（平行）

plt.tight_layout()
plt.show()

---
## Part 2：面積 = 行列式！連結回 Ch5

### 理論

這裡是 3Blue1Brown 把前面章節串起來的關鍵時刻。

Ch5 說：行列式 = 線性變換對面積的縮放因子。
現在反過來看：**兩個向量排成矩陣的 column → det 就是它們圍出的帶符號面積**。

$$\vec{v} \times \vec{w} = \det \begin{bmatrix} v_1 & w_1 \\ v_2 & w_2 \end{bmatrix}$$

為什麼？因為把 $\hat{i}=(1,0)$ 和 $\hat{j}=(0,1)$ 通過矩陣 $[\vec{v}\;\vec{w}]$ 變換後，
單位正方形（面積=1）就變成了 $\vec{v}$ 和 $\vec{w}$ 圍成的平行四邊形。
面積的縮放倍數 = det = 帶符號面積。

In [ ]:
# === 實驗 2：面積 = det，視覺化單位正方形的變換 ===

v = np.array([3, 1])
w = np.array([1, 2])
M = np.column_stack([v, w])  # v 和 w 當作 column

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左圖：單位正方形
ax = axes[0]
square = np.array([[0,0],[1,0],[1,1],[0,1]])
ax.add_patch(plt.Polygon(square, alpha=0.3, color='skyblue', edgecolor='blue', lw=2))
ax.annotate('', xy=[1,0], xytext=(0,0), arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.annotate('', xy=[0,1], xytext=(0,0), arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.text(0.5, -0.3, 'î', fontsize=14, color='red', fontweight='bold', ha='center')
ax.text(-0.3, 0.5, 'ĵ', fontsize=14, color='blue', fontweight='bold', ha='center')
ax.text(0.5, 0.5, 'Area = 1', fontsize=13, ha='center', va='center', fontweight='bold')
ax.set_xlim(-1, 4); ax.set_ylim(-1, 4); ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_title("Before: 單位正方形\nî → î,  ĵ → ĵ", fontsize=12)

# 右圖：變換後 = 平行四邊形
ax = axes[1]
transformed = (M @ square.T).T
det_val = np.linalg.det(M)
ax.add_patch(plt.Polygon(transformed, alpha=0.3, color='salmon', edgecolor='red', lw=2))
ax.annotate('', xy=v, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.annotate('', xy=w, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
ax.text(v[0]*0.5+0.2, v[1]*0.5, f'v=î→({v[0]},{v[1]})', fontsize=11, color='red', fontweight='bold')
ax.text(w[0]*0.5-0.5, w[1]*0.5+0.2, f'w=ĵ→({w[0]},{w[1]})', fontsize=11, color='blue', fontweight='bold')
center = (v + w) / 2
ax.text(center[0], center[1], f'Area = |det| = {abs(det_val):.0f}',
        fontsize=13, ha='center', va='center', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlim(-1, 5); ax.set_ylim(-1, 4); ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_title(f"After: î→v, ĵ→w\ndet = {det_val:.0f}", fontsize=12)

plt.tight_layout()
plt.show()

print(f"矩陣 M = [v | w] =")
print(M)
print(f"det(M) = {v[0]}×{w[1]} - {v[1]}×{w[0]} = {det_val:.0f}")
print(f"→ 單位正方形的面積 1 被縮放了 {abs(det_val):.0f} 倍")
print(f"→ 這就是 v 和 w 圍成的平行四邊形面積！")

---
## Part 3：3D 叉積 — 結果是一個向量！

### 理論

這是點積和叉積最大的差異：

| | 點積 (Dot Product) | 叉積 (Cross Product) |
|---|---|---|
| 輸入 | 兩個向量 | 兩個 **3D** 向量 |
| 輸出 | 一個**純量** | 一個**向量** |
| 意義 | 方向相似度 | 垂直於兩者的新方向 |
| 大小 | $\|\vec{a}\|\|\vec{b}\|\cos\theta$ | $\|\vec{a}\|\|\vec{b}\|\sin\theta$ = 平行四邊形面積 |

叉積 $\vec{v} \times \vec{w}$ 的結果向量：
- **大小** = $\vec{v}$ 和 $\vec{w}$ 圍成的平行四邊形面積
- **方向** = 垂直於 $\vec{v}$ 和 $\vec{w}$ 所在的平面，由**右手定則**決定

In [ ]:
# === 實驗 3：3D 叉積的視覺化 ===

def plot_3d_cross(v, w, elev=25, azim=45):
    """完整視覺化 3D 叉積：兩向量、平行四邊形、叉積向量"""
    v, w = np.array(v, dtype=float), np.array(w, dtype=float)
    cross = np.cross(v, w)
    area = np.linalg.norm(cross)
    
    fig = plt.figure(figsize=(10, 9))
    ax = fig.add_subplot(111, projection='3d')
    
    # 畫三個向量
    origin = [0, 0, 0]
    ax.quiver(*origin, *v, color='red', arrow_length_ratio=0.1, lw=2.5, label=f'v = {v.tolist()}')
    ax.quiver(*origin, *w, color='blue', arrow_length_ratio=0.1, lw=2.5, label=f'w = {w.tolist()}')
    ax.quiver(*origin, *cross, color='purple', arrow_length_ratio=0.08, lw=3, 
              label=f'v×w = {np.round(cross, 2).tolist()}')
    
    # 畫平行四邊形 (v, w 圍��的)
    parallelogram = np.array([origin, v, v+w, w])
    poly = Poly3DCollection([parallelogram], alpha=0.25, facecolor='orange', edgecolor='gray', lw=1.5)
    ax.add_collection3d(poly)
    
    # 畫從叉積向量到平行四邊形的虛線（強調垂直）
    ax.plot([0, cross[0]], [0, cross[1]], [0, cross[2]], 'purple', lw=1, ls='--', alpha=0.5)
    
    # 標記
    ax.text(cross[0]*1.1, cross[1]*1.1, cross[2]*1.1, f'v×w\n|v×w|={area:.2f}',
            fontsize=11, color='purple', fontweight='bold')
    
    # 設定
    all_pts = np.array([origin, v.tolist(), w.tolist(), cross.tolist(), (v+w).tolist()])
    max_range = np.abs(all_pts).max() + 1
    ax.set_xlim(-max_range, max_range)
    ax.set_ylim(-max_range, max_range)
    ax.set_zlim(-max_range, max_range)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.view_init(elev=elev, azim=azim)
    ax.legend(fontsize=10, loc='upper left')
    ax.set_title(f"v × w：大小 = 平行四邊形面積 = {area:.2f}\n方向 = 垂直於 v 和 w（右手定則）", fontsize=13)
    
    plt.tight_layout()
    plt.show()
    
    # 驗證垂直性
    print(f"v × w = {cross}")
    print(f"|v × w| = {area:.4f} (= 平行四邊形面積)")
    print(f"\n驗證垂直：")
    print(f"  (v×w) · v = {np.dot(cross, v):.10f}  ← 應為 0")
    print(f"  (v×w) · w = {np.dot(cross, w):.10f}  ← 應為 0")
    print(f"  → 叉積結果確實垂直於 v 和 w！")

plot_3d_cross([2, 0, 0], [0, 3, 0])  # 最簡單的例子：x軸 × y軸 = z軸方向

In [ ]:
# === 試一個更有趣的例子 ===
print("═" * 50)
print("一般性的 3D 向量：")
print("═" * 50)
plot_3d_cross([1, 2, 0], [0, 1, 3], elev=20, azim=135)

---
## Part 4：右手定則 — 叉積方向怎麼判斷

### 理論

垂直於一個平面有**兩個方向**（上和下），叉積選哪一個？

**右手定則**：
1. 右手四指從 $\vec{v}$ 的方向「捲」向 $\vec{w}$
2. 大拇指指的方向就是 $\vec{v} \times \vec{w}$

**關鍵性質**：$\vec{v} \times \vec{w} = -(\vec{w} \times \vec{v})$

交換順序 → 方向反轉！（叉積是**反交換**的）

In [ ]:
# === 實驗 4：v×w vs w×v — 反交換性 ===

v = np.array([1, 2, 0])
w = np.array([0, 1, 3])

vxw = np.cross(v, w)
wxv = np.cross(w, v)

print(f"v = {v}")
print(f"w = {w}")
print(f"")
print(f"v × w = {vxw}")
print(f"w × v = {wxv}")
print(f"")
print(f"v×w + w×v = {vxw + wxv}  ← 零向量！互為相反數")
print(f"v×w == -(w×v) ? {np.array_equal(vxw, -wxv)}")

# 視覺化
fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')
ax1.quiver(0,0,0, *v, color='red', arrow_length_ratio=0.1, lw=2, label='v')
ax1.quiver(0,0,0, *w, color='blue', arrow_length_ratio=0.1, lw=2, label='w')
ax1.quiver(0,0,0, *vxw, color='purple', arrow_length_ratio=0.08, lw=3, label=f'v×w = {vxw.tolist()}')
ax1.set_xlim(-4,4); ax1.set_ylim(-4,4); ax1.set_zlim(-4,7)
ax1.set_title("v × w（右手：v → w）", fontsize=12)
ax1.legend(); ax1.view_init(20, 135)

ax2 = fig.add_subplot(122, projection='3d')
ax2.quiver(0,0,0, *v, color='red', arrow_length_ratio=0.1, lw=2, label='v')
ax2.quiver(0,0,0, *w, color='blue', arrow_length_ratio=0.1, lw=2, label='w')
ax2.quiver(0,0,0, *wxv, color='darkgreen', arrow_length_ratio=0.08, lw=3, label=f'w×v = {wxv.tolist()}')
ax2.set_xlim(-4,4); ax2.set_ylim(-4,4); ax2.set_zlim(-7,4)
ax2.set_title("w × v = −(v × w)（方向反轉！）", fontsize=12)
ax2.legend(); ax2.view_init(20, 135)

plt.tight_layout()
plt.show()

---
## Part 5：3D 叉積的計算公式 — 「假行列式」

### 理論

3Blue1Brown 把叉積公式寫成一個 3×3 行列式的樣子：

$$\vec{v} \times \vec{w} = \det \begin{bmatrix} \hat{i} & v_1 & w_1 \\ \hat{j} & v_2 & w_2 \\ \hat{k} & v_3 & w_3 \end{bmatrix}$$

這是一個「假的」行列式，因為第一列放的是**向量**而不是數字。
但如果你照正常的 cofactor expansion 展開：

$$= \hat{i}(v_2 w_3 - v_3 w_2) - \hat{j}(v_1 w_3 - v_3 w_1) + \hat{k}(v_1 w_2 - v_2 w_1)$$

你就得到了叉積的三個分量。

每一個分量其實是一個 **2×2 的行列式**（也就是 2D 叉積！）：
- $x$ 分量 = 把兩向量「投影」到 yz 平面後的面積
- $y$ 分量 = 把兩向量「投影」到 xz 平面後的面積（注意負號）
- $z$ 分量 = 把兩向量「投影」到 xy 平面後的面積

In [ ]:
# === 實驗 5a：手算叉積 vs np.cross ===

v = np.array([1, 2, 3])
w = np.array([4, 5, 6])

# 手算：cofactor expansion
# x 分量 = det([[v2, w2], [v3, w3]]) = v2*w3 - v3*w2
x = v[1]*w[2] - v[2]*w[1]
# y 分量 = -det([[v1, w1], [v3, w3]]) = -(v1*w3 - v3*w1)
y = -(v[0]*w[2] - v[2]*w[0])
# z 分量 = det([[v1, w1], [v2, w2]]) = v1*w2 - v2*w1
z = v[0]*w[1] - v[1]*w[0]

manual = np.array([x, y, z])
numpy_result = np.cross(v, w)

print(f"v = {v}")
print(f"w = {w}")
print()
print("手算過程：")
print(f"  x = v₂w₃ - v₃w₂ = {v[1]}×{w[2]} - {v[2]}×{w[1]} = {x}")
print(f"  y = -(v₁w₃ - v₃w₁) = -({v[0]}×{w[2]} - {v[2]}×{w[0]}) = {y}")
print(f"  z = v₁w₂ - v₂w₁ = {v[0]}×{w[1]} - {v[1]}×{w[0]} = {z}")
print()
print(f"手算結果：  {manual}")
print(f"np.cross：  {numpy_result}")
print(f"一致？ {np.array_equal(manual, numpy_result)}")

In [ ]:
# === 實驗 5b：每個分量 = 投影到座標平面的 2D 叉積（面積）===

v = np.array([1, 2, 3])
w = np.array([4, 5, 6])
cross = np.cross(v, w)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

projections = [
    ("yz 平面 (去掉 x)", [1, 2], [1, 2], 'x', 'Y', 'Z'),  # 取 yz 分量
    ("xz 平面 (去掉 y)", [0, 2], [0, 2], 'y', 'X', 'Z'),   # 取 xz 分量
    ("xy 平面 (去掉 z)", [0, 1], [0, 1], 'z', 'X', 'Y'),    # 取 xy 分量
]

for idx, (title, v_idx, w_idx, comp, xlabel, ylabel) in enumerate(projections):
    ax = axes[idx]
    
    v2d = v[v_idx]
    w2d = w[w_idx]
    area_2d = v2d[0]*w2d[1] - v2d[1]*w2d[0]
    
    # 畫平行四邊形
    para = np.array([[0,0], v2d, v2d+w2d, w2d])
    color = 'lightcoral' if area_2d >= 0 else 'lightblue'
    ax.add_patch(plt.Polygon(para, alpha=0.3, color=color, edgecolor='gray', lw=1.5))
    
    ax.annotate('', xy=v2d, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.annotate('', xy=w2d, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    
    # y 分量有額外的負號
    sign = -1 if comp == 'y' else 1
    actual_comp = sign * area_2d
    
    sign_str = f"−det = −({area_2d})" if comp == 'y' else f"det = {area_2d}"
    
    center = (v2d + w2d) / 2
    ax.text(center[0], center[1], f'det = {area_2d}', fontsize=12, ha='center',
            fontweight='bold', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    lim = max(np.abs(np.concatenate([v2d, w2d, v2d+w2d]))) + 1.5
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(f"投影到 {title}\n→ cross 的 {comp} 分量 = {sign_str} = {actual_comp}",
                 fontsize=11)

plt.tight_layout()
plt.show()

print(f"v × w = {cross}")
print(f"  x 分量 =  det(yz) = {v[1]*w[2]-v[2]*w[1]}")
print(f"  y 分量 = -det(xz) = {-(v[0]*w[2]-v[2]*w[0])}")
print(f"  z 分量 =  det(xy) = {v[0]*w[1]-v[1]*w[0]}")

---
## Part 6：叉積的重要性質 & 動手實驗

### 性質整理

1. **反交換**：$\vec{v} \times \vec{w} = -(\vec{w} \times \vec{v})$
2. **垂直性**：$(\vec{v} \times \vec{w}) \cdot \vec{v} = 0$ 且 $(\vec{v} \times \vec{w}) \cdot \vec{w} = 0$
3. **大小 = 面積**：$|\vec{v} \times \vec{w}| = |\vec{v}||\vec{w}|\sin\theta$
4. **平行 → 零**：$\vec{v} \times \vec{v} = \vec{0}$（跟自己叉 = 面積為零）
5. **分配律**：$\vec{u} \times (\vec{v} + \vec{w}) = \vec{u} \times \vec{v} + \vec{u} \times \vec{w}$

### 叉積 vs 點積速查

| | 點積 $\vec{a} \cdot \vec{b}$ | 叉積 $\vec{a} \times \vec{b}$ |
|---|---|---|
| 結果 | 純量 | 向量 |
| 含 cos 還是 sin | $\cos\theta$ | $\sin\theta$ |
| = 0 時 | 垂直 | 平行 |
| 交換性 | $\vec{a} \cdot \vec{b} = \vec{b} \cdot \vec{a}$ | $\vec{a} \times \vec{b} = -\vec{b} \times \vec{a}$ |

In [ ]:
# === 實驗 6a：驗證所有性質 ===

v = np.array([1, 2, 3])
w = np.array([4, 5, 6])
u = np.array([2, -1, 1])

cross_vw = np.cross(v, w)
print("v =", v, " w =", w, " u =", u)
print("v × w =", cross_vw)
print()

# 性質 1：反交換
print("1. 反交換：v×w == -(w×v)?", np.array_equal(cross_vw, -np.cross(w, v)))

# 性質 2：垂直
print(f"2. 垂直：  (v×w)·v = {np.dot(cross_vw, v)},  (v×w)·w = {np.dot(cross_vw, w)}")

# 性質 3：|v×w| = |v||w|sinθ
cos_theta = np.dot(v, w) / (np.linalg.norm(v) * np.linalg.norm(w))
sin_theta = np.sqrt(1 - cos_theta**2)
expected_mag = np.linalg.norm(v) * np.linalg.norm(w) * sin_theta
print(f"3. 大小：  |v×w| = {np.linalg.norm(cross_vw):.6f},  |v||w|sinθ = {expected_mag:.6f}")

# 性質 4：v × v = 0
print(f"4. 自叉：  v×v = {np.cross(v, v)}")

# 性質 5：分配律
lhs = np.cross(u, v + w)
rhs = np.cross(u, v) + np.cross(u, w)
print(f"5. 分��律：u×(v+w) = {lhs},  u×v + u×w = {rhs},  一致？{np.array_equal(lhs, rhs)}")

In [ ]:
# === 實驗 6b：sin vs cos — 點積和叉積如何互補 ===

# 固定 v，讓 w 繞著轉，觀察 dot 和 cross 的大小如何變化

v_fixed = np.array([1, 0, 0])

angles_deg = np.arange(0, 361, 1)
dot_vals = []
cross_mags = []

for deg in angles_deg:
    rad = np.radians(deg)
    w_rot = np.array([np.cos(rad), np.sin(rad), 0])
    dot_vals.append(np.dot(v_fixed, w_rot))
    cross_mags.append(np.linalg.norm(np.cross(v_fixed, w_rot)))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(angles_deg, dot_vals, 'b-', lw=2.5, label='|dot(v, w)| = cos θ  (方向相似度)')
ax.plot(angles_deg, cross_mags, 'r-', lw=2.5, label='|v × w| = sin θ  (平行四邊形面積)')
ax.fill_between(angles_deg, dot_vals, alpha=0.1, color='blue')
ax.fill_between(angles_deg, cross_mags, alpha=0.1, color='red')

# 標記關鍵角度
for deg, event in [(0, '平行\ndot=1\ncross=0'), (90, '垂直\ndot=0\ncross=1'), 
                    (180, '反平行\ndot=-1\ncross=0')]:
    ax.axvline(deg, color='gray', ls=':', lw=1)
    ax.text(deg+3, 0.7, event, fontsize=9, color='gray')

ax.set_xlabel('θ (degrees)', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('點積 (cos) vs 叉積大小 (sin)：完美互補', fontsize=14)
ax.set_xticks([0, 45, 90, 135, 180, 225, 270, 315, 360])
ax.legend(fontsize=11, loc='lower right')
ax.axhline(0, color='k', lw=0.5)
plt.tight_layout()
plt.show()

print("θ=0°:   完全同方向 → dot 最大, cross = 0 (面積為零，因為重疊)")
print("θ=90°:  完全垂直   → dot = 0,  cross 最大 (面積最大)")
print("θ=180°: 完全反方向 → dot 最小, cross = 0 (又重疊了)")
print()
print("口訣：dot 測「同不同方向」(cos)，cross 測「有多垂直」(sin)")

In [ ]:
# === 練習區：改 v 和 w 看 3D ��積怎麼變 ===

my_v = [1, 0, 2]    # ← 改這裡
my_w = [0, 3, 1]    # ← 改這裡

plot_3d_cross(my_v, my_w, elev=25, azim=120)

---
## 總結

| 概念 | 2D | 3D |
|---|---|---|
| 叉積結果 | 純量（帶符號面積） | 向量（面積 + 方向） |
| 計算 | $v_1 w_2 - v_2 w_1 = \det[v \| w]$ | 「假行列式」cofactor 展開 |
| 幾何意義 | 平行四邊形面積 | 平行四邊形面積 + 垂直方向 |
| 正負/方向 | 正=逆時針, 負=順時針 | 右手定則 |

**與行列式的深層連結**：
- 2D 叉積 = 2×2 行列式 = ch5 的面積縮放因子
- 3D 叉積的每個分量 = 一個 2×2 行列式（投影到座標平面的面積）
- 下一章（8-2）會從「線性變換 + 對偶性」的角度重新理解叉積公式